# Aula 01 — Do modelo linear ao neurônio artificial

Este laboratório constrói um neurônio escalar **somente com NumPy**. Vamos executar o
*forward*, derivar o *backward*, verificar o gradiente por diferenças finitas, treinar
uma porta OR e produzir duas contraprovas: XOR não é linearmente separável e empilhar
camadas afins sem ativação continua sendo apenas uma transformação afim.

**Dependências mínimas:** Python 3.11, NumPy 1.26, Matplotlib 3.8 e nbformat 5.9.
Não há download, segredo ou credencial. Todos os dados são sintéticos e explícitos.


## 1. Contrato experimental e convenções

- exemplos ocupam linhas: $X\in\mathbb{R}^{n\times d}$;
- um neurônio tem $W\in\mathbb{R}^{d\times1}$ e $b\in\mathbb{R}^{1}$;
- o resultado $Z=XW+b$ e a probabilidade $P=\sigma(Z)$ têm shape $(n,1)$;
- a perda é a média da BCE sobre $n$ exemplos;
- `SEED = 20260909` fixa qualquer inicialização pseudoaleatória;
- o teste numérico usa diferenças centrais e **não** participa do treinamento.


In [ ]:
from __future__ import annotations

import platform

import matplotlib
import matplotlib.pyplot as plt
import numpy as np

SEED = 20260909
rng = np.random.default_rng(SEED)
np.set_printoptions(precision=6, suppress=True)

print({
    "python": platform.python_version(),
    "numpy": np.__version__,
    "matplotlib": matplotlib.__version__,
    "seed": SEED,
})


## 2. Um exemplo escalar resolvido

Para $x=[2,-1]^\top$, $w=[0{,}5,-2]^\top$ e $b=-0{,}5$:

$$z=w^\top x+b=0{,}5(2)+(-2)(-1)-0{,}5=2{,}5.$$

Usaremos ReLU $a=\max(0,z)$ e a perda quadrática $L=\frac12(a-y)^2$ com
$y=1{,}5$. Como $z>0$, a derivada local da ReLU vale 1.


In [ ]:
x = np.array([2.0, -1.0])
w = np.array([0.5, -2.0])
b = -0.5
y = 1.5

z = float(w @ x + b)
a = max(0.0, z)
loss = 0.5 * (a - y) ** 2

dL_da = a - y
da_dz = float(z > 0.0)
dL_dz = dL_da * da_dz
grad_w = dL_dz * x
grad_b = dL_dz
grad_x = dL_dz * w

assert np.isclose(z, 2.5) and np.isclose(loss, 0.5)
assert np.allclose(grad_w, [2.0, -1.0])
assert np.isclose(grad_b, 1.0)
assert np.allclose(grad_x, [0.5, -2.0])
print({"z": z, "a": a, "L": loss})
print({"dL_dw": grad_w, "dL_db": grad_b, "dL_dx": grad_x})


### Atualização local e aproximação de primeira ordem

Com taxa $\eta=0{,}1$, atualizamos parâmetros na direção oposta ao gradiente. A
aproximação $\Delta L\approx\nabla L^\top\Delta\theta$ só é exata no limite de passos
pequenos; comparar previsão e valor real é uma auditoria útil da implementação.


In [ ]:
eta = 0.1
w_new = w - eta * grad_w
b_new = b - eta * grad_b
z_new = float(w_new @ x + b_new)
loss_new = 0.5 * (max(0.0, z_new) - y) ** 2

delta_theta = np.r_[w_new - w, b_new - b]
gradient_theta = np.r_[grad_w, grad_b]
predicted_delta = float(gradient_theta @ delta_theta)
actual_delta = loss_new - loss

assert loss_new < loss
print({
    "L_antes": round(loss, 6),
    "L_depois": round(loss_new, 6),
    "delta_previsto_1a_ordem": round(predicted_delta, 6),
    "delta_real": round(actual_delta, 6),
})


## 3. Forward vetorizado, BCE estável e backward

Para classificação binária, o neurônio calcula logits $Z=XW+b$ e probabilidades
$P=\sigma(Z)$. A BCE com logits é implementada como

$$\frac1n\sum_i\left[\max(z_i,0)-y_i z_i+\log(1+e^{-|z_i|})\right],$$

forma equivalente à BCE usual, mas estável para logits muito grandes. Sua derivada é
$dZ=(P-y)/n$. Pela regra da cadeia:

$$dW=X^\top dZ,\qquad db=\sum_i dZ_i,\qquad dX=dZ W^\top.$$


In [ ]:
def sigmoid(z: np.ndarray) -> np.ndarray:
    z = np.asarray(z, dtype=float)
    out = np.empty_like(z)
    positive = z >= 0
    out[positive] = 1.0 / (1.0 + np.exp(-z[positive]))
    exp_z = np.exp(z[~positive])
    out[~positive] = exp_z / (1.0 + exp_z)
    return out


def bce_with_logits(z: np.ndarray, y: np.ndarray) -> float:
    terms = np.maximum(z, 0.0) - z * y + np.log1p(np.exp(-np.abs(z)))
    return float(np.mean(terms))


def forward(X: np.ndarray, W: np.ndarray, b: np.ndarray):
    assert X.ndim == 2 and W.shape == (X.shape[1], 1) and b.shape == (1,)
    Z = X @ W + b
    P = sigmoid(Z)
    assert Z.shape == P.shape == (X.shape[0], 1)
    return Z, P


def loss_and_grads(X: np.ndarray, y: np.ndarray, W: np.ndarray, b: np.ndarray):
    Z, P = forward(X, W, b)
    assert y.shape == Z.shape
    loss = bce_with_logits(Z, y)
    dZ = (P - y) / X.shape[0]
    dW = X.T @ dZ
    db = dZ.sum(axis=0)
    dX = dZ @ W.T
    assert dW.shape == W.shape and db.shape == b.shape and dX.shape == X.shape
    return loss, {"W": dW, "b": db, "X": dX}, P


extreme = sigmoid(np.array([-1000.0, 0.0, 1000.0]))
assert np.array_equal(extreme, np.array([0.0, 0.5, 1.0]))
assert np.isfinite(bce_with_logits(np.array([[-1000.0], [1000.0]]), np.array([[0.0], [1.0]])))
print("sigmoid([-1000, 0, 1000]) =", extreme)


## 4. Dados explícitos: porta OR

As quatro linhas abaixo formam toda a população lógica. Não há split: o objetivo é
verificar cálculo e capacidade representacional, não estimar generalização estatística.


In [ ]:
X_or = np.array([
    [0.0, 0.0],
    [0.0, 1.0],
    [1.0, 0.0],
    [1.0, 1.0],
])
y_or = np.array([[0.0], [1.0], [1.0], [1.0]])

W0 = rng.normal(0.0, 0.2, size=(2, 1))
b0 = np.zeros(1)
initial_loss, initial_grads, initial_p = loss_and_grads(X_or, y_or, W0, b0)

assert X_or.shape == (4, 2) and y_or.shape == (4, 1)
assert np.isfinite(initial_loss)
print({"W0": W0.ravel(), "b0": b0, "loss_inicial": round(initial_loss, 8)})
print("probabilidades iniciais:", initial_p.ravel())


## 5. Gradient checking por diferenças centrais

Para cada parâmetro $\theta_j$, aproximamos

$$\frac{\partial L}{\partial\theta_j}\approx
\frac{L(\theta_j+h)-L(\theta_j-h)}{2h}.$$

O teste detecta erro de implementação; ele não prova que a derivação ou o modelo são
adequados. O erro relativo usa denominador pelo menos 1 para permanecer bem definido
perto de gradientes nulos.


In [ ]:
def pack(W: np.ndarray, b: np.ndarray) -> np.ndarray:
    return np.r_[W.ravel(), b.ravel()]


def unpack(theta: np.ndarray, d: int):
    return theta[:d].reshape(d, 1), theta[d:].reshape(1)


def numerical_gradient(X, y, W, b, h=1e-5):
    theta = pack(W, b)
    numerical = np.zeros_like(theta)
    for j in range(theta.size):
        plus, minus = theta.copy(), theta.copy()
        plus[j] += h
        minus[j] -= h
        Wp, bp = unpack(plus, X.shape[1])
        Wm, bm = unpack(minus, X.shape[1])
        Lp = loss_and_grads(X, y, Wp, bp)[0]
        Lm = loss_and_grads(X, y, Wm, bm)[0]
        numerical[j] = (Lp - Lm) / (2.0 * h)
    return numerical


analytical = pack(initial_grads["W"], initial_grads["b"])
numerical = numerical_gradient(X_or, y_or, W0, b0)
relative = np.abs(analytical - numerical) / np.maximum(
    1.0, np.maximum(np.abs(analytical), np.abs(numerical))
)
max_gradient_error = float(relative.max())

assert max_gradient_error < 1e-9
print("analítico:", analytical)
print("numérico:  ", numerical)
print(f"erro relativo máximo = {max_gradient_error:.3e}")


### O passo $h$ também é um hiperparâmetro numérico

Passos grandes sofrem erro de truncamento; passos minúsculos sofrem cancelamento e
arredondamento. A curva abaixo procura uma faixa saudável, sem escolher $h$ para
melhorar o modelo.


In [ ]:
hs = np.logspace(-2, -11, 10)
h_errors = []
for h in hs:
    numeric_h = numerical_gradient(X_or, y_or, W0, b0, h=h)
    err_h = np.max(np.abs(analytical - numeric_h) / np.maximum(1.0, np.abs(analytical)))
    h_errors.append(float(err_h))

best_h = float(hs[int(np.argmin(h_errors))])
best_h_error = float(np.min(h_errors))
assert np.isfinite(h_errors).all() and best_h_error < 1e-8

fig, ax = plt.subplots(figsize=(7, 4))
ax.loglog(hs, h_errors, marker="o")
ax.set(xlabel="passo h", ylabel="erro relativo máximo",
       title="Diferenças centrais: truncamento versus arredondamento")
ax.grid(True, which="both", alpha=0.3)
plt.show()
print({"melhor_h_na_grade": best_h, "erro": best_h_error})


## 6. Treinamento por gradiente descendente

Agora repetimos *forward*, *backward* e atualização simultânea de $W$ e $b$. Guardamos
a perda antes de cada atualização. O critério de classe é $P\ge0{,}5$.


In [ ]:
def train_neuron(X, y, *, seed=SEED, learning_rate=0.8, steps=2500):
    local_rng = np.random.default_rng(seed)
    W = local_rng.normal(0.0, 0.2, size=(X.shape[1], 1))
    b = np.zeros(1)
    history = []
    for _ in range(steps):
        loss, grads, _ = loss_and_grads(X, y, W, b)
        history.append(loss)
        W -= learning_rate * grads["W"]
        b -= learning_rate * grads["b"]
    final_loss, _, probabilities = loss_and_grads(X, y, W, b)
    predictions = (probabilities >= 0.5).astype(int)
    return W, b, np.asarray(history), final_loss, probabilities, predictions


W_or, b_or, history_or, final_loss_or, p_or, pred_or = train_neuron(X_or, y_or)
accuracy_or = float(np.mean(pred_or == y_or))

assert final_loss_or < initial_loss
assert accuracy_or == 1.0
assert np.all(np.diff(history_or[100:]) <= 1e-12)
print({
    "W": W_or.ravel(), "b": b_or,
    "loss_final": round(final_loss_or, 8),
    "accuracy": accuracy_or,
})
print("probabilidades OR:", p_or.ravel())


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(history_or)
axes[0].set(xlabel="passo", ylabel="BCE com logits", title="Convergência na porta OR")
axes[0].grid(alpha=0.3)

colors = np.where(y_or.ravel() == 1, "tab:orange", "tab:blue")
axes[1].scatter(X_or[:, 0], X_or[:, 1], c=colors, s=100, edgecolor="black")
xline = np.linspace(-0.2, 1.2, 100)
yline = -(W_or[0, 0] * xline + b_or[0]) / W_or[1, 0]
axes[1].plot(xline, yline, "k--", label="z = 0 (p = 0,5)")
axes[1].set(xlabel="$x_1$", ylabel="$x_2$", title="Fronteira linear aprendida",
            xlim=(-0.2, 1.2), ylim=(-0.2, 1.2))
axes[1].legend()
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()


## 7. Auditoria de $dX$ por derivada direcional

Embora entradas não sejam parâmetros deste neurônio, $dX$ será necessário quando uma
camada enviar gradiente à camada anterior. Uma direção aleatória $v$ deve satisfazer
$\frac{d}{d\epsilon}L(X+\epsilon v)|_{0}=\langle dX,v\rangle$.


In [ ]:
loss_or, grads_or, _ = loss_and_grads(X_or, y_or, W_or, b_or)
direction = rng.normal(size=X_or.shape)
direction /= np.linalg.norm(direction)
h = 1e-5
directional_numeric = (
    loss_and_grads(X_or + h * direction, y_or, W_or, b_or)[0]
    - loss_and_grads(X_or - h * direction, y_or, W_or, b_or)[0]
) / (2.0 * h)
directional_analytic = float(np.sum(grads_or["X"] * direction))
directional_error = abs(directional_numeric - directional_analytic)

assert directional_error < 1e-9
print({
    "direcional_analítica": directional_analytic,
    "direcional_numérica": directional_numeric,
    "erro_absoluto": directional_error,
})


## 8. Contraprova 1: um neurônio não resolve XOR

XOR atribui a mesma classe a vértices opostos do quadrado. Nenhuma reta separa seus
positivos dos negativos. Treinar com várias seeds verifica o sintoma empírico, mas a
razão é geométrica: o classificador continua sendo um semiespaço definido por $z=0$.


In [ ]:
X_xor = X_or.copy()
y_xor = np.array([[0.0], [1.0], [1.0], [0.0]])
xor_runs = []
for seed in range(20):
    W_x, b_x, hist_x, loss_x, p_x, pred_x = train_neuron(
        X_xor, y_xor, seed=seed, learning_rate=0.5, steps=3000
    )
    xor_runs.append((loss_x, float(np.mean(pred_x == y_xor)), W_x, b_x, p_x))

best_xor_accuracy = max(run[1] for run in xor_runs)
best_xor_loss = min(run[0] for run in xor_runs)
assert best_xor_accuracy < 1.0
assert best_xor_loss >= np.log(2.0) - 1e-8
print({
    "melhor_accuracy_em_20_seeds": best_xor_accuracy,
    "menor_BCE": round(best_xor_loss, 8),
    "log_2": round(float(np.log(2.0)), 8),
})


## 9. Contraprova 2: duas camadas afins colapsam em uma

Sem ativação entre camadas,
$$(XW_1+b_1)W_2+b_2=X(W_1W_2)+(b_1W_2+b_2).$$
Logo, profundidade sem não linearidade não amplia a família de funções representáveis.


In [ ]:
X_demo = rng.normal(size=(8, 3))
W1 = rng.normal(size=(3, 5))
b1 = rng.normal(size=(1, 5))
W2 = rng.normal(size=(5, 2))
b2 = rng.normal(size=(1, 2))

two_layers = (X_demo @ W1 + b1) @ W2 + b2
W_collapsed = W1 @ W2
b_collapsed = b1 @ W2 + b2
one_layer = X_demo @ W_collapsed + b_collapsed
collapse_error = float(np.max(np.abs(two_layers - one_layer)))

assert two_layers.shape == one_layer.shape == (8, 2)
assert collapse_error < 1e-12
print({"shape": two_layers.shape, "erro_máximo_do_colapso": collapse_error})


## 10. Verificações automáticas consolidadas

Uma verificação cobre um contrato específico: shapes, finitude, derivadas, capacidade
na OR, limite na XOR e equivalência algébrica. Passar nesses testes não elimina riscos
de dados, objetivo ou implantação fora deste laboratório.


In [ ]:
checks = {
    "shapes_forward_backward": initial_grads["X"].shape == X_or.shape,
    "sigmoid_extremos_finita": bool(np.isfinite(extreme).all()),
    "gradient_check_parametros": max_gradient_error < 1e-9,
    "gradient_check_entrada": directional_error < 1e-9,
    "OR_aprendida": accuracy_or == 1.0,
    "XOR_nao_separada": best_xor_accuracy < 1.0,
    "camadas_afins_colapsam": collapse_error < 1e-12,
}
assert all(checks.values())
print(checks)
print(f"{sum(checks.values())}/{len(checks)} contratos satisfeitos")


## 11. Leituras dos resultados e próximos passos

1. O neurônio é uma composição: transformação afim, ativação e perda.
2. O backward reaproveita valores do forward e propaga derivadas locais.
3. Shapes fazem parte da especificação, não são um detalhe de implementação.
4. BCE com logits e sigmoid por ramos evitam instabilidade sem mascarar o erro.
5. Gradient checking testa o código; não substitui derivação, protocolo ou avaliação.
6. OR cabe em uma fronteira linear; XOR exige uma representação não linear.

Na **Aula 02**, formalizaremos o perceptron e sua regra de atualização por erro. O MLP
e o backpropagation completo aparecem depois, conforme a grade canônica da trilha.
